# BIST local LLM: train, evaluate, use (Google Colab)

Runtime: **T4 GPU** (Runtime -> Change runtime type). Run the cells top to bottom. Training takes a few minutes for Qwen3-1.7B and longer for Qwen3-4B.

What the model learns (the course sample test): turn a question into a JSON **tool plan** (`rank_sector_relative_strength`, `test_weekday_pattern`, `detect_reversal_events`, `calculate_fundamental_change`, `get_macro_context`, ...), write an **evidence-bound English summary** from the tool output, and **refuse** look-ahead, unknown tickers and investment advice. Calculation, the quality gate, the backtest and the final status are done by code (`src/llm/agent.py`), never by the model.

The training data is generated in this notebook from the real tools (`python -m src.llm.dataset`). Targets are deterministic (tool outputs + `summaries.py`), so no number in a target can be wrong; six tickers are held out for the evaluation.


In [ ]:
BRANCH = "zeynep"   # the branch that holds the current code; it must be pushed to GitHub first
!git clone https://github.com/zeyneptanrivermis/bist-harness.git
%cd /content/bist-harness
!git fetch origin
!git switch $BRANCH
!git pull origin $BRANCH

In [ ]:
!pip install -q -r requirements-llm.txt

In [ ]:
import os
os.environ["BIST_LLM_MODEL"] = "Qwen/Qwen3-4B"   # demo model; "Qwen/Qwen3-1.7B" is the faster fallback
ADAPTER = "artifacts/llm/qwen_bist_lora/adapter"
os.environ["BIST_LLM_ADAPTER"] = ADAPTER
print(os.environ["BIST_LLM_MODEL"], ADAPTER)

In [ ]:
# Data is not in Git. Colab has internet, so build everything here (about 5-10 minutes the first time).
!python -m src.data_pipeline          # 30 stocks + XU100 prices from Yahoo into data/processed
!python -m src.macro --download       # USD/TRY and EUR/TRY into data/macro
# TCMB policy rate, TUIK CPI and Fed funds need official files: upload data/macro/TCMB_POLICY_RATE.csv etc. (columns date,value,published_date)
# here if the team has them; without them the tools report those series as unavailable and the model must not state values.
!python -m src.llm.dataset --output-dir data/llm/dataset_v3 --per-family 80 --reports-per-family 36

## Train

In [ ]:
import json, random, os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from src.llm.training_data import stable_split, validate_records

SEED = 481
random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select a T4 GPU runtime (Runtime -> Change runtime type)."
MODEL_NAME = os.environ["BIST_LLM_MODEL"]
OUTPUT_DIR = Path(ADAPTER).parent

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

reviewed_records = read_jsonl("data/llm/dataset_v3/records.jsonl")
validate_records(reviewed_records, require_human_review=False)
manifest = json.load(open("data/llm/dataset_v3/manifest.json"))
print(f"{len(reviewed_records)} records {manifest['by_task']} -> {OUTPUT_DIR}")
print("report statuses:", manifest["report_status"], "| skipped:", len(manifest["skipped"]))

In [ ]:
# QLoRA training (about 700 records x 3 epochs; 1.7B is much faster than 4B on a T4)
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from src.llm.training_data import stable_split

splits = stable_split(reviewed_records)
train_dataset = Dataset.from_list(splits['train'] + splits['test'])  # evaluate.py has its own held-out questions.remove_columns(['metadata'])
validation_dataset = Dataset.from_list(splits['validation']).remove_columns(['metadata'])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
# torch reports bf16 as supported on T4 (emulated); real support needs compute capability >= 8
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization, dtype=DTYPE, device_map={'': 0})
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM', target_modules='all-linear')  # all linear layers: copying long evidence needs more than q/v
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    max_length=3072,  # evidence-JSON prompts are ~1.5-2k tokens; 1024 would cut off the answer
    gradient_checkpointing=True,
    bf16=USE_BF16,
    fp16=not USE_BF16,  # T4 has no bf16
    report_to='none',
    seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
# fp16 GradScaler cannot unscale bf16 grads, so keep every trainable (LoRA) weight in fp32
print('trainable dtypes before:', {str(p.dtype) for p in trainer.model.parameters() if p.requires_grad}, '| model dtype:', model.dtype)
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'adapter'))
print('Saved adapter to', OUTPUT_DIR / 'adapter')

In [ ]:
# Colab runtimes reset. Copy the adapter to Google Drive right after training (also download results.zip at the end).
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/bist_llm && cp -r $ADAPTER /content/drive/MyDrive/bist_llm/
!ls /content/drive/MyDrive/bist_llm

## Use and evaluate
`run_agent(question, chat_fn)` is the full loop: the model writes a JSON tool plan, the harness runs the quality gate, the tools and the backtest, the model writes the summary, and a guard checks it. The evaluation asks 12 questions in the style of the course sample test (6 with tickers never seen in training) twice each, for the base model and the adapter, and writes the rates (invalid/wrong tool plan, numeric hallucination, unsupported conclusion, look-ahead, missing evidence, rule violations, reproducibility) to `artifacts/llm/eval_report.json`.

In [ ]:
from src.llm.agent import local_chat_fn, run_agent
chat_fn = local_chat_fn(ADAPTER)
q = "Does THYAO show a 'Monday–Tuesday–Wednesday down, then Thursday–Friday up' pattern over 01.01.2024 – 31.12.2025? Does the pattern still hold over 01.01.2026 – 10.09.2026?"
result = run_agent(q, chat_fn)
print(result["tool_call"], result["violations"], sep="\n"); print(result["answer"])

In [ ]:
%cd /content/bist-harness
!git pull
!python -m src.llm.evaluate --adapter $ADAPTER

In [ ]:
!zip -qr results.zip $ADAPTER artifacts/llm/eval_report.json data/llm/dataset_v3/manifest.json
from google.colab import files; files.download("results.zip")